# AnchorKV: T4 receiver-head trace collection

This notebook performs the first real-model AnchorKV experiment on a Google Colab T4. It generates with efficient SDPA, replays only bounded completed traces with eager attention, and saves compact sentence-level statistics. It does **not** yet claim physical KV-cache compression or latency improvements.

In [ ]:
import os

if not os.path.exists('/content/Dynamic-Quantization'):
    !git clone https://github.com/Dev-Sinha13/Dynamic-Quantization.git /content/Dynamic-Quantization
%cd /content/Dynamic-Quantization
!python -m pip install -q -e ".[research]"

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({gpu_gib:.1f} GiB)')
if 'T4' not in gpu_name:
    print('Warning: this notebook was calibrated for a T4; record the actual GPU in your results.')

In [ ]:
from anchorkv import ModelGeometry, estimate_eager_capture

geometry = ModelGeometry(
    parameters=600_000_000, layers=28, query_heads=16, kv_heads=8, head_dim=64
)
estimate = estimate_eager_capture(geometry, sequence_length=768, available_gib=gpu_gib)
estimate.as_gib()

In [ ]:
from huggingface_hub import model_info
from anchorkv.hf import HFTraceConfig

MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_REVISION = model_info(MODEL_ID).sha
CONFIG = HFTraceConfig(
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    max_sequence_length=768,
    max_new_tokens=128,
    seed=7,
)
print('Pinned model revision:', MODEL_REVISION)

In [ ]:
PROMPTS = [
    'Solve this carefully and show concise reasoning in complete sentences: If 3 notebooks cost $12, how much do 7 notebooks cost at the same rate?',
    'Solve this carefully and show concise reasoning in complete sentences: A train travels 180 miles in 3 hours. At the same speed, how far does it travel in 5 hours?',
    'Solve this carefully and show concise reasoning in complete sentences: Maria has twice as many marbles as Lee. Together they have 36 marbles. How many does each person have?',
]

In [ ]:
import gc
from pathlib import Path
from anchorkv.hf import extract_hf_trace

output_dir = Path('artifacts/colab')
output_dir.mkdir(parents=True, exist_ok=True)
artifact_paths = []
run_summaries = []
for index, prompt in enumerate(PROMPTS):
    sample_id = f'math-{index:03d}'
    result = extract_hf_trace(
        prompt, sample_id=sample_id, output_path=output_dir / sample_id, config=CONFIG
    )
    artifact_paths.append(result.artifact_path)
    run_summaries.append({
        'sample_id': sample_id,
        'sequence_length': result.sequence_length,
        'peak_gpu_gib': result.peak_gpu_bytes / 1024**3,
        'generated_text': result.generated_text,
    })
    print(run_summaries[-1])
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from anchorkv import load_trace
from anchorkv.artifacts import trace_summary

[trace_summary(load_trace(path)) for path in artifact_paths]

In [ ]:
from anchorkv.analysis import discover_from_paths, save_manifest

manifest = discover_from_paths(artifact_paths, top_k=16)
manifest_path = save_manifest(manifest, output_dir / 'receiver-heads.json')
manifest['receiver_heads'][:5]

In [ ]:
import matplotlib.pyplot as plt

top_heads = manifest['receiver_heads'][:10]
labels = [f"L{head['layer']}:H{head['query_head']}" for head in top_heads]
scores = [head['ranking_score'] for head in top_heads]
plt.figure(figsize=(10, 4))
plt.bar(labels, scores)
plt.ylabel('kurtosis × stability')
plt.title('AnchorKV receiver-head candidates')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Save the run

Download these artifacts before the Colab runtime expires. The next milestone will use them to select and causally suppress candidate thought-anchor sentences.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/anchorkv-colab-traces', 'zip', output_dir)
files.download(archive)